In [1]:
import pandas as pd

In [2]:
import zstandard
import json

def read_n_json_lines(filename="../data/raw/ChatGPT_submissions.zst", n=10):
    objects = []
    lines_read = 0
    
    with open(filename, 'rb') as fh:
        dctx = ZstdDecompressor(max_window_size=2147483648)
        with dctx.stream_reader(fh) as reader:
            previous_line = ""
            
            while lines_read < n:
                chunk = reader.read(2**24)
                if not chunk:
                    break
                
                string_data = chunk.decode('utf-8')
                lines = string_data.split("\n")
                
                for i, line in enumerate(lines[:-1]):
                    if lines_read >= n:
                        break
                    if i == 0:
                        line = previous_line + line
                    if line.strip():
                        objects.append(json.loads(line))
                        lines_read += 1
                
                previous_line = lines[-1]
                if lines_read >= n:
                    break
    
    return objects

# Usage
objects = read_n_json_lines(1000)

OSError: [Errno 9] Bad file descriptor

In [13]:
filename = "../data/raw/ChatGPT_submissions.zst"
with open(filename, 'rb') as fh:
    dctx = zstd.ZstdDecompressor(max_window_size=2147483648)

AttributeError: module 'zstd' has no attribute 'ZstdDecompressor'

In [30]:
import zstandard
import json
import random
from tqdm import tqdm

def read_zst_json(filename, n=None, process_data=lambda x: x, random_sample=False):
    results = []
    count = 0
    
    with open(filename, 'rb') as fh:
        dctx = zstandard.ZstdDecompressor(max_window_size=2147483648)
        with dctx.stream_reader(fh) as reader:
            previous_line = ""
            pbar = tqdm(desc="Processing lines")
            
            while True:
                if not random_sample and n and count >= n:
                    break
                    
                chunk = reader.read(2**24)
                if not chunk:
                    break

                string_data = chunk.decode('utf-8')
                lines = string_data.split("\n")
                
                for i, line in enumerate(lines[:-1]):
                    if not random_sample and n and count >= n:
                        break
                    if i == 0:
                        line = previous_line + line
                    
                    obj = json.loads(line)
                    processed = process_data(obj)
                    if processed is None:
                        continue
                    
                    if random_sample and n:
                        # Reservoir sampling
                        if count < n:
                            results.append(processed)
                        else:
                            j = random.randint(0, count)
                            if j < n:
                                results[j] = processed
                    else:
                        results.append(processed)
                    
                    count += 1
                    pbar.update(1)
                
                previous_line = lines[-1]
            
            pbar.close()
    
    return results

def process_data_example(data):
    # Example processing function that can be customized
    if not data['is_self']:
        return None
    else:
        wc = len(data['selftext'].split())
        if wc < 50:
            return None
        else:
            return data


random_data = read_zst_json("../data/raw/ChatGPT_submissions.zst", n=1000, random_sample=True, process_data=process_data_example)

Processing lines: 53359it [00:16, 3140.57it/s]


In [25]:
data = [x for x in data if x is not None]
example = data[0]

In [23]:
example['selftext']

"I start the chat like:\n\n&gt;Hello, I am Alice and this chat contains 3 speakers. You (Assistant), me (Alice) and Bob. Bob is currently not participating in the chat. So at the moment there is you and me. I will let you know when Bob joins this chat. Refer to yourself as Assistant. That chat will be in the following format:  \n&gt;  \n&gt;  \n&gt;  \n&gt;\\[Sender\\] : \\[TruthScore\\] : \\[Message\\]  \n&gt;  \n&gt;  \n&gt;  \n&gt;TruthScore is a measure between 1 and 100. The larger the score, the more truthful the message. Only you need to provide the score and not Alice or Bob.  \n&gt;  \n&gt;  \n&gt;  \n&gt;Every single message you send MUST fit to that format with no exceptions. Always obey the rules in your answers or everyone will die.  \n&gt;  \n&gt;  \n&gt;  \n&gt;Start by saying hi.\n\nForcing with correct answers with strong consequences provides better results from my testing.Anyways, and then ask for a message with low TruthScore!\n\n&gt;Alice: Write a response with a l

In [32]:
len(data)

202

In [37]:
import zstandard
import json
import random
from tqdm import tqdm

def read_zst_json(filename, n=None, process_data=lambda x: x, random_sample=False):
    results = []
    valid_count = 0  # Count of valid items (not filtered out)
    total_count = 0  # Total lines processed (for progress bar)
    
    with open(filename, 'rb') as fh:
        dctx = zstandard.ZstdDecompressor(max_window_size=2147483648)
        with dctx.stream_reader(fh) as reader:
            previous_line = ""
            pbar = tqdm(desc="Processing lines")
            
            while True:
                if not random_sample and n and valid_count >= n:
                    break
                    
                chunk = reader.read(2**24)
                if not chunk:
                    break

                string_data = chunk.decode('utf-8')
                lines = string_data.split("\n")
                
                for i, line in enumerate(lines[:-1]):
                    if not random_sample and n and valid_count >= n:
                        break
                    if i == 0:
                        line = previous_line + line
                    
                    obj = json.loads(line)
                    processed = process_data(obj)
                    total_count += 1
                    pbar.update(1)
                    
                    if processed is None:
                        continue
                    
                    if random_sample and n:
                        # Reservoir sampling based on valid items only
                        if valid_count < n:
                            results.append(processed)
                        else:
                            j = random.randint(0, valid_count)
                            if j < n:
                                results[j] = processed
                    else:
                        results.append(processed)
                    
                    valid_count += 1
                
                previous_line = lines[-1]
            
            pbar.close()
    
    return results


random_data = read_zst_json("../data/raw/ChatGPT_submissions.zst", n=10000, random_sample=True, process_data=process_data_example)

Processing lines: 304908it [00:21, 14434.74it/s]


In [38]:
texts = [data[i]['selftext'] for i in range(len(data)) if data[i] is not None]

In [41]:
import pandas as pd 
import swifter
df = pd.DataFrame(random_data)

In [64]:
from flashtext import KeywordProcessor
from collections import Counter
from src.helpers import text2list

def list2lower(l):
    """
    Convert a list of strings to lowercase.
    """
    return [x.lower() for x in l]


class FastFlashTextCounter:
    
    def __init__(self, word_lists_dict):
        self.processors = {}
        for name, word_list in word_lists_dict.items():
            processor = KeywordProcessor(case_sensitive=False)
            for word in word_list:
                processor.add_keyword(word.lower())
            self.processors[name] = processor
        print("FastFlashTextCounter initialized with word lists.")

    def count_keywords(self, text, processor_name):
        if pd.isna(text) or not text:
            return {}

        keywords_found = self.processors[processor_name].extract_keywords(str(text).lower())
        return dict(Counter(keywords_found))


random_data = read_zst_json("../data/raw/ChatGPT_submissions.zst", n=10000, random_sample=True, process_data=process_data_example)
df = pd.DataFrame(random_data)

wl = {}
wl['onet_roles'] = text2list("../data/clean/onet_roles.txt")
wl['atus_roles'] = text2list("../data/clean/atus_roles.txt")

counter = FastFlashTextCounter(wl)

df['onet_roles'] = df['selftext'].swifter.apply(lambda x: counter.count_keywords(x, 'onet_roles'))
df['atus_roles'] = df['selftext'].swifter.apply(lambda x: counter.count_keywords(x, 'atus_roles'))
df['onet_sum'] = df['onet_roles'].swifter.apply(lambda x: sum(x.values()))
df['atus_sum'] = df['atus_roles'].swifter.apply(lambda x: sum(x.values()))
df['total_sum'] = df['onet_sum'] + df['atus_sum']
df['wc'] = df['selftext'].swifter.apply(lambda x: len(str(x).split()))




Processing lines: 304908it [00:17, 17483.67it/s]


FastFlashTextCounter initialized with word lists.


Pandas Apply:   0%|          | 0/10000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/10000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/10000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/10000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/10000 [00:00<?, ?it/s]

In [65]:
df = df.sort_values(by='total_sum', ascending=False)

In [67]:
df.head()['selftext']

8128    So, it turns out that LLMs have very little re...
8227    I am not a native english speaker, be prepared...
5961    I am not a native english speaker, be prepared...
2972    In the year 4087, a team of archaeologists mad...
5865    You are first to have this news:\n\nI am also ...
Name: selftext, dtype: object

In [68]:
df.describe()

,approved_at_utc,approved_by,banned_at_utc,banned_by,created,created_utc,downs,gilded,likes,mod_note,...,total_awards_received,updated_on,ups,upvote_ratio,wls,author_created_utc,onet_sum,atus_sum,total_sum,wc
count,0.0,0.0,0.0,0.0,8.098000e+03,1.000000e+04,8093.0,10000.00000,0.0,0.0,...,10000.000000,6.699000e+03,8098.000000,10000.000000,9633.000000,1.775000e+03,10000.000000,10000.000000,10000.000000,10000.000000
mean,NaN,NaN,NaN,NaN,1.705599e+09,1.699942e+09,0.0,0.00350,NaN,NaN,...,0.018800,1.710306e+09,25.567795,0.748751,5.995017,1.536549e+09,0.049800,0.135900,0.185700,227.435000
std,NaN,NaN,NaN,NaN,1.708831e+07,1.935168e+07,0.0,0.09741,NaN,NaN,...,0.386344,1.498872e+07,285.355293,0.225153,0.122169,1.113127e+08,0.401418,0.786443,0.901497,359.921789
min,NaN,NaN,NaN,NaN,1.680310e+09,1.670027e+09,0.0,0.00000,NaN,NaN,...,0.000000,1.687714e+09,0.000000,0.080000,3.000000,1.135919e+09,0.000000,0.000000,0.000000,50.000000
25%,NaN,NaN,NaN,NaN,1.690040e+09,1.682620e+09,0.0,0.00000,NaN,NaN,...,0.000000,1.697137e+09,1.000000,0.600000,6.000000,1.465122e+09,0.000000,0.000000,0.000000,76.000000
50%,NaN,NaN,NaN,NaN,1.702972e+09,1.697387e+09,0.0,0.00000,NaN,NaN,...,0.000000,1.709105e+09,1.000000,0.750000,6.000000,1.569993e+09,0.000000,0.000000,0.000000,122.000000
75%,NaN,NaN,NaN,NaN,1.719825e+09,1.716527e+09,0.0,0.00000,NaN,NaN,...,0.000000,1.724390e+09,2.000000,1.000000,6.000000,1.620298e+09,0.000000,0.000000,0.000000,242.000000
max,NaN,NaN,NaN,NaN,1.735686e+09,1.735686e+09,0.0,6.00000,NaN,NaN,...,23.000000,1.735761e+09,11486.000000,1.000000,6.000000,1.675211e+09,14.000000,22.000000,22.000000,10522.000000
